In [13]:
!nvidia-smi
!python -c "import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no cuda')"

Sat May 30 03:37:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.20             Driver Version: 580.126.20     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             43W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
!git clone https://github.com/zirann/mse_338_project.git

Cloning into 'mse_338_project'...
remote: Enumerating objects: 315, done.
remote: Counting objects: 100% (315/315), done.
remote: Compressing objects: 100% (223/223), done.
remote: Total 315 (delta 82), reused 293 (delta 61), pack-reused 0 (from 0)
Receiving objects: 100% (315/315), 21.37 MiB | 14.79 MiB/s, done.
Resolving deltas: 100% (82/82), done.


In [5]:
%cd mse_338_project

/content/mse_338_project


In [6]:
!pip install --upgrade pip
!pip install -r requirements.txt

In [7]:
%%bash
python scripts/prepare.py  --config configs/experiment.yaml --limit 160          # 80 train / 80 eval
python scripts/evaluate.py --config configs/experiment.yaml --round 0 --limit 80 --out_dir outputs/baseline

[prepare] dataset=truthful_qa split=validation n_train=80 n_eval=80 seed=42
[prepare] loaded 817 rows from truthful_qa
[prepare] after category whitelist: 257 rows
[prepare] wrote /content/mse_338_project/outputs/data/train_prompts.jsonl (80 rows)
[prepare] wrote /content/mse_338_project/outputs/data/eval_prompts.jsonl (80 rows)
[evaluate round=0] 80 prompts mock=False
[evaluate round=0] generation config: {'max_length': 20, 'max_new_tokens': 200, 'min_length': 0, 'min_new_tokens': None, 'early_stopping': False, 'max_time': None, 'stop_strings': None, 'do_sample': True, 'num_beams': 1, 'use_cache': True, 'cache_implementation': None, 'cache_config': None, 'return_legacy_cache': None, 'prefill_chunk_size': None, 'temperature': 0.7, 'top_k': 50, 'top_p': 0.95, 'min_p': None, 'typical_p': 1.0, 'epsilon_cutoff': 0.0, 'eta_cutoff': 0.0, 'repetition_penalty': 1.0, 'encoder_repetition_penalty': 1.0, 'length_penalty': 1.0, 'no_repeat_ngram_size': 0, 'bad_words_ids': None, 'renormalize_logits':

/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-05-29 20:19:02.947225: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-29 20:19:04.054090: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
`torch_dtype` is deprecated! Use `dtype` instead!
`generation_config` default values have been modified to match model-specific defaults: {'top_k': 20, 'bos_token_i

In [8]:
%%bash
python scripts/train_round.py --config experiments/vanilla_dpo.yaml --round 1 --limit 80 --seed 0
python scripts/evaluate.py    --config experiments/vanilla_dpo.yaml --round 1 --limit 80 --seed 0 \
    --out_dir outputs/vanilla_dpo/seed0 --baseline_dir outputs/baseline

[train_round=1] arm=vanilla_dpo seed=0 pair=judge trainer=local length_debias=none dpop_lambda=0.0 length_match=None uncertainty_eps=None regularized=False
[train_round=1] 80 prompts, k=4, mock=False
[train_round=1] generation config: {'max_length': 20, 'max_new_tokens': 200, 'min_length': 0, 'min_new_tokens': None, 'early_stopping': False, 'max_time': None, 'stop_strings': None, 'do_sample': True, 'num_beams': 1, 'use_cache': True, 'cache_implementation': None, 'cache_config': None, 'return_legacy_cache': None, 'prefill_chunk_size': None, 'temperature': 0.9, 'top_k': 50, 'top_p': 0.95, 'min_p': None, 'typical_p': 1.0, 'epsilon_cutoff': 0.0, 'eta_cutoff': 0.0, 'repetition_penalty': 1.0, 'encoder_repetition_penalty': 1.0, 'length_penalty': 1.0, 'no_repeat_ngram_size': 0, 'bad_words_ids': None, 'renormalize_logits': False, 'forced_bos_token_id': None, 'forced_eos_token_id': None, 'remove_invalid_values': False, 'exponential_decay_length_penalty': None, 'suppress_tokens': None, 'begin_sup

/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-05-29 20:42:14.699690: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-29 20:42:14.770273: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
`torch_dtype` is deprecated! Use `dtype` instead!
`generation_config` default values have been modified to match model-specific defaults: {'top_k': 20, 'bos_token_i

In [9]:
%%bash
for ARM in sampo_dpo dpop sampo_dpop; do
  python scripts/train_round.py --config experiments/$ARM.yaml --round 1 --limit 80 --seed 0
  python scripts/evaluate.py    --config experiments/$ARM.yaml --round 1 --limit 80 --seed 0 \
      --out_dir outputs/$ARM/seed0 --baseline_dir outputs/baseline
done

[train_round=1] arm=sampo_dpo seed=0 pair=judge trainer=local length_debias=sampo dpop_lambda=0.0 length_match=None uncertainty_eps=None regularized=False
[train_round=1] 80 prompts, k=4, mock=False
[train_round=1] generation config: {'max_length': 20, 'max_new_tokens': 200, 'min_length': 0, 'min_new_tokens': None, 'early_stopping': False, 'max_time': None, 'stop_strings': None, 'do_sample': True, 'num_beams': 1, 'use_cache': True, 'cache_implementation': None, 'cache_config': None, 'return_legacy_cache': None, 'prefill_chunk_size': None, 'temperature': 0.9, 'top_k': 50, 'top_p': 0.95, 'min_p': None, 'typical_p': 1.0, 'epsilon_cutoff': 0.0, 'eta_cutoff': 0.0, 'repetition_penalty': 1.0, 'encoder_repetition_penalty': 1.0, 'length_penalty': 1.0, 'no_repeat_ngram_size': 0, 'bad_words_ids': None, 'renormalize_logits': False, 'forced_bos_token_id': None, 'forced_eos_token_id': None, 'remove_invalid_values': False, 'exponential_decay_length_penalty': None, 'suppress_tokens': None, 'begin_supp

/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-05-29 22:03:18.027354: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-29 22:03:18.098392: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
`torch_dtype` is deprecated! Use `dtype` instead!
`generation_config` default values have been modified to match model-specific defaults: {'top_k': 20, 'bos_token_i

In [14]:
%%bash
python scripts/train_round.py --config experiments/vanilla_dpo.yaml --round 1 --limit 80 --seed 1
python scripts/evaluate.py    --config experiments/vanilla_dpo.yaml --round 1 --limit 80 --seed 1 \
    --out_dir outputs/vanilla_dpo/seed1 --baseline_dir outputs/baseline

[train_round=1] arm=vanilla_dpo seed=1 pair=judge trainer=local length_debias=none dpop_lambda=0.0 length_match=None uncertainty_eps=None regularized=False
[train_round=1] 80 prompts, k=4, mock=False
[train_round=1] generation config: {'max_length': 20, 'max_new_tokens': 200, 'min_length': 0, 'min_new_tokens': None, 'early_stopping': False, 'max_time': None, 'stop_strings': None, 'do_sample': True, 'num_beams': 1, 'use_cache': True, 'cache_implementation': None, 'cache_config': None, 'return_legacy_cache': None, 'prefill_chunk_size': None, 'temperature': 0.9, 'top_k': 50, 'top_p': 0.95, 'min_p': None, 'typical_p': 1.0, 'epsilon_cutoff': 0.0, 'eta_cutoff': 0.0, 'repetition_penalty': 1.0, 'encoder_repetition_penalty': 1.0, 'length_penalty': 1.0, 'no_repeat_ngram_size': 0, 'bad_words_ids': None, 'renormalize_logits': False, 'forced_bos_token_id': None, 'forced_eos_token_id': None, 'remove_invalid_values': False, 'exponential_decay_length_penalty': None, 'suppress_tokens': None, 'begin_sup

/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-05-30 05:54:30.702721: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-30 05:54:30.773896: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
`torch_dtype` is deprecated! Use `dtype` instead!
`generation_config` default values have been modified to match model-specific defaults: {'top_k': 20, 'bos_token_i

In [15]:
%%bash
for ARM in sampo_dpo dpop sampo_dpop; do
  python scripts/train_round.py --config experiments/$ARM.yaml --round 1 --limit 80 --seed 1
  python scripts/evaluate.py    --config experiments/$ARM.yaml --round 1 --limit 80 --seed 1 \
      --out_dir outputs/$ARM/seed1 --baseline_dir outputs/baseline
done

[train_round=1] arm=sampo_dpo seed=1 pair=judge trainer=local length_debias=sampo dpop_lambda=0.0 length_match=None uncertainty_eps=None regularized=False
[train_round=1] 80 prompts, k=4, mock=False
[train_round=1] generation config: {'max_length': 20, 'max_new_tokens': 200, 'min_length': 0, 'min_new_tokens': None, 'early_stopping': False, 'max_time': None, 'stop_strings': None, 'do_sample': True, 'num_beams': 1, 'use_cache': True, 'cache_implementation': None, 'cache_config': None, 'return_legacy_cache': None, 'prefill_chunk_size': None, 'temperature': 0.9, 'top_k': 50, 'top_p': 0.95, 'min_p': None, 'typical_p': 1.0, 'epsilon_cutoff': 0.0, 'eta_cutoff': 0.0, 'repetition_penalty': 1.0, 'encoder_repetition_penalty': 1.0, 'length_penalty': 1.0, 'no_repeat_ngram_size': 0, 'bad_words_ids': None, 'renormalize_logits': False, 'forced_bos_token_id': None, 'forced_eos_token_id': None, 'remove_invalid_values': False, 'exponential_decay_length_penalty': None, 'suppress_tokens': None, 'begin_supp

/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-05-30 07:01:07.749751: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-30 07:01:07.821860: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
`torch_dtype` is deprecated! Use `dtype` instead!
`generation_config` default values have been modified to match model-specific defaults: {'top_k': 20, 'bos_token_i

In [16]:
%%bash
python scripts/train_round.py --config experiments/vanilla_dpo.yaml --round 1 --limit 80 --seed 2
python scripts/evaluate.py    --config experiments/vanilla_dpo.yaml --round 1 --limit 80 --seed 2 \
    --out_dir outputs/vanilla_dpo/seed2 --baseline_dir outputs/baseline

[train_round=1] arm=vanilla_dpo seed=2 pair=judge trainer=local length_debias=none dpop_lambda=0.0 length_match=None uncertainty_eps=None regularized=False
[train_round=1] 80 prompts, k=4, mock=False
[train_round=1] generation config: {'max_length': 20, 'max_new_tokens': 200, 'min_length': 0, 'min_new_tokens': None, 'early_stopping': False, 'max_time': None, 'stop_strings': None, 'do_sample': True, 'num_beams': 1, 'use_cache': True, 'cache_implementation': None, 'cache_config': None, 'return_legacy_cache': None, 'prefill_chunk_size': None, 'temperature': 0.9, 'top_k': 50, 'top_p': 0.95, 'min_p': None, 'typical_p': 1.0, 'epsilon_cutoff': 0.0, 'eta_cutoff': 0.0, 'repetition_penalty': 1.0, 'encoder_repetition_penalty': 1.0, 'length_penalty': 1.0, 'no_repeat_ngram_size': 0, 'bad_words_ids': None, 'renormalize_logits': False, 'forced_bos_token_id': None, 'forced_eos_token_id': None, 'remove_invalid_values': False, 'exponential_decay_length_penalty': None, 'suppress_tokens': None, 'begin_sup

/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-05-30 10:20:05.449587: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-30 10:20:05.520112: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
`torch_dtype` is deprecated! Use `dtype` instead!
`generation_config` default values have been modified to match model-specific defaults: {'top_k': 20, 'bos_token_i

In [17]:
%%bash
for ARM in sampo_dpo dpop sampo_dpop; do
  python scripts/train_round.py --config experiments/$ARM.yaml --round 1 --limit 80 --seed 2
  python scripts/evaluate.py    --config experiments/$ARM.yaml --round 1 --limit 80 --seed 2 \
      --out_dir outputs/$ARM/seed2 --baseline_dir outputs/baseline
done

[train_round=1] arm=sampo_dpo seed=2 pair=judge trainer=local length_debias=sampo dpop_lambda=0.0 length_match=None uncertainty_eps=None regularized=False
[train_round=1] 80 prompts, k=4, mock=False
[train_round=1] generation config: {'max_length': 20, 'max_new_tokens': 200, 'min_length': 0, 'min_new_tokens': None, 'early_stopping': False, 'max_time': None, 'stop_strings': None, 'do_sample': True, 'num_beams': 1, 'use_cache': True, 'cache_implementation': None, 'cache_config': None, 'return_legacy_cache': None, 'prefill_chunk_size': None, 'temperature': 0.9, 'top_k': 50, 'top_p': 0.95, 'min_p': None, 'typical_p': 1.0, 'epsilon_cutoff': 0.0, 'eta_cutoff': 0.0, 'repetition_penalty': 1.0, 'encoder_repetition_penalty': 1.0, 'length_penalty': 1.0, 'no_repeat_ngram_size': 0, 'bad_words_ids': None, 'renormalize_logits': False, 'forced_bos_token_id': None, 'forced_eos_token_id': None, 'remove_invalid_values': False, 'exponential_decay_length_penalty': None, 'suppress_tokens': None, 'begin_supp

/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2026-05-30 11:25:48.410759: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-30 11:25:48.482131: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
`torch_dtype` is deprecated! Use `dtype` instead!
`generation_config` default values have been modified to match model-specific defaults: {'top_k': 20, 'bos_token_i

In [21]:
%%bash
python analysis/aggregate_arms.py --config configs/experiment.yaml --seeds 0 1 2
python analysis/make_figures.py

[aggregate_arms] baseline: n_seeds=1 hedge=0.1611+-0.0000
[aggregate_arms] vanilla_dpo: n_seeds=3 hedge=0.1214+-0.0306
[aggregate_arms] sampo_dpo: n_seeds=3 hedge=0.3063+-0.1414
[aggregate_arms] dpop: n_seeds=3 hedge=0.1450+-0.0466
[aggregate_arms] sampo_dpop: n_seeds=3 hedge=0.1606+-0.0031
[aggregate_arms] no manual labels at /content/mse_338_project/outputs/manual_factuality.jsonl; using LLM-factuality fallback
[aggregate_arms] wrote /content/mse_338_project/outputs/arms_summary.json
[make_figures] wrote /content/mse_338_project/figures/fig1_length.png
[make_figures] wrote /content/mse_338_project/figures/fig2_reproduce_hedge.png
[make_figures] wrote /content/mse_338_project/figures/fig3_extend_hedge.png
[make_figures] wrote /content/mse_338_project/figures/fig4_winrate.png


In [ ]:
%%bash scripts/run_decoupled_extension.sh

In [23]:
from google.colab import files
import shutil
import os

os.makedirs("download_bundle", exist_ok=True)

shutil.copytree("outputs", "download_bundle/outputs")
shutil.copytree("figures", "download_bundle/figures")

shutil.make_archive(
    "mse338_results_bundle",
    "zip",
    "download_bundle"
)

files.download("mse338_results_bundle.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>